In [0]:
from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
spark

In [0]:
print(spark)

In [0]:
%sql
SELECT *
FROM csv.`/Volumes/labuser14944459_1777641927/raw/data`

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/labuser14944459_1777641927/raw/data/BigMart Sales.csv")

In [0]:
display(df)

## Scanning Optimization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
## Turn off AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/labuser14944459_1777641927/raw/data/BigMart Sales.csv")

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
# Change the default partition size to 128KB
spark.conf.set("spark.sql.files.maxPartitionBytes","131072")

In [0]:
# Change the default partition size to 128MB
spark.conf.set("spark.sql.files.maxPartitionBytes","134217728")

In [0]:
# Get no. of partitions
df.rdd.getNumPartitions()

In [0]:
# Repartition
df = df.repartition(10)

In [0]:
df.rdd.getNumPartitions()

We need partitions so that we can apply parallelisim

In [0]:
# Get partition info

df.withColumn("partition_id", spark_partition_id()).groupBy("partition_id").count().display()

In [0]:
df.write.format("parquet")\
    .mode("overwrite")\
    .save("/Volumes/labuser14944459_1777641927/raw/data/files/")

In [0]:
df_new = spark.read.format("parquet")\
    .load("/Volumes/labuser14944459_1777641927/raw/data/files/")
display(df_new)

In [0]:
df_new = df_new.filter(df_new['Outlet_location_Type'] == 'Tier 1')
display(df_new)

## Scanning Optimization

In [0]:
df.write.format("parquet")\
  .mode("overwrite")\
  .partitionBy("Outlet_location_Type")\
  .save("/Volumes/labuser14944459_1777641927/raw/data/files")

In [0]:
df.select("Outlet_Location_Type").distinct().display()

In [0]:
df_new = spark.read.format("parquet")\
    .load("/Volumes/labuser14944459_1777641927/raw/data/files/")

df_new = df_new.filter(df_new['Outlet_location_Type'] == 'Tier 1')
display(df_new)

In [0]:
display(df_new)

## Join Optimization